# Standard dynamic analysis with HERMESS

*PowerUp 2026, hands-on: the same study in code (about 10 minutes)*

HERMESS simulates power systems written as nonlinear differential-algebraic
equations (DAEs). The desktop GUI just ran a study on a small system: pick it,
set the options, run, read the modes. This notebook reproduces that session in
one call, then covers what the code adds, one function at a time:

1. the shipped systems and the two text files that define one,
2. the GUI session in one call,
3. a local copy of a system, ready to edit,
4. a run with our own settings,
5. the one-line diagram,
6. any quantity by name, plotted,
7. numbers out (metrics, CSV),
8. a different disturbance, and two runs compared,
9. quasi-static versus dynamic network in one flag,
10. eigenvalues and participation factors in one call,
11. a slider on a droop gain and an inertia,
12. the command line, and results that travel.

Everything called here is `hermess.analysis`, the package's notebook layer.
Each function shows its full argument list with `plot?` (or Shift+Tab inside
the brackets), and the docs page "Analysis and plotting" carries the same
material; the table at the end maps every function used here to its section.

## 0. Setup

On Colab, the first cell installs `hermess` from PyPI (about a minute); that is
the entire setup. Locally, run inside the workshop environment
(`uv run jupyter lab`) and it does nothing.

In [ ]:
import importlib.metadata, importlib.util, os, subprocess, sys

if importlib.util.find_spec("hermess") is None:                      # Colab
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "hermess>=1.7.1"])
    # Colab preimports numpy; if pip just upgraded it, the kernel holds a stale
    # mix of old and new files. A one-time restart fixes it; the install sticks.
    stale = [m for m in ("numpy", "pandas", "matplotlib") if m in sys.modules
             and importlib.metadata.version(m) != sys.modules[m].__version__]
    if stale:
        print("pip upgraded", ", ".join(stale), "under the running kernel;")
        print("restarting the runtime now. When it reconnects (a few seconds),")
        print("run the cells again from the top; the install is already done.")
        os.kill(os.getpid(), 9)

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hermess
from hermess.analysis import *

print("hermess", hermess.__version__)
assert tuple(int(x) for x in hermess.__version__.split(".")[:3]) >= (1, 7, 1), \
    "this notebook needs hermess 1.7.1 or newer: pip install -U hermess"

`hermess.analysis` draws in whatever matplotlib style is active and imposes no
solver settings, so this cell holds the two notebook-side pieces: the ETH plot
style, and `WS`, the run settings every simulation in this notebook shares.
`run` is `simulate` with those settings pre-filled. Delete this cell and
everything still works, in matplotlib's default style and hermess's default
configuration.

In [ ]:
ETH = {"blue": "#215CAF", "petrol": "#007894", "green": "#627313", "bronze": "#8E6713",
       "red": "#B7352D", "purple": "#A7117A", "grey": "#6F6F6F"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[ETH[k] for k in
        ("blue", "petrol", "red", "green", "bronze", "purple", "grey")]),
    "font.family": "serif",
    "font.serif": ["Latin Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm", "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 110, "legend.frameon": False,
})

WS = dict(line_dyn=True, ts=0.001, incl_lim=False, quiet=True,       # the workshop's run settings
          int_scheme_sim_options={"reltol": 1e-8, "abstol": 1e-10,
                                  "max_num_steps": 100000, "jit": False})

def run(system, **overrides):
    """simulate() with the workshop settings pre-filled; any Config field overrides."""
    return simulate(system, **{**WS, **overrides})

## 1. Shipped systems and where they live

The package ships 39 ready-made systems (`list_systems()`).

 The shipped systems live in the hermess system directory.

Every system lives in a folder with two files:

* `sim_param.txt`: devices (one row each, `Class, idx = ..., bus = ..., params`),
  network branches (`Line, ...`), and the bus data used by the initial power flow
  (`BusInit, ...`).
* `sim_dist.txt`: the disturbances, one row each, with a time and a type.


For this tutorial, we use a three-bus
system: a synchronous machine (slack) at bus 1, a constant impedance load at
bus 2, and a droop grid-forming converter at bus 3.

```
     bus 1 ─────────── bus 2 ─────────── bus 3
       │                 │                 │
      SG1              load              GFMI2
  synchronous        (impedance)      grid-forming
    machine                             converter
       └───────────────────────────────────┘
```

## 2. The GUI session, in one call

The GUI (`hermess-gui`, installed with `pip install "hermess[gui]"`) is a front end
over the same package: its systems tree is `hermess.list_systems()`, its options
dialog is generated from the `Config` object, Run calls `hermess.simulate()` in a
worker process, and its builder writes ordinary system folders that this notebook
runs unchanged. So the session you watched can be reproduced here. One call runs
the system, prints the initial power flow (the Power flow tab) and the modal
report (the Small signal tab); the Time domain tab is the returned object,
plotted, which is what the rest of this notebook does.

## 3. Setting up your workflow

Copy the systems you would like to use into a local directory and edit them there
before simulating.

`copy_system` copies the folder out of the installed package into a local
`systems/` directory, so you can edit it freely (calling it again leaves your
edits alone). The GUI's builder produces the same kind of folder.

Things to notice:

* the class name in the first column selects the model (`SynchronousSubtransientSP`,
  `GridForming`, `StaticZIP`, `Line`, `BusInit`),
* parameters you leave out take the model's defaults,
* `BusInit` is only for the power flow that initializes the simulation at a steady state;
  the dynamic models are then initialized to match it (no "startup transient"),
* the disturbance is a +10 MW load step at bus 2 after 1 s.

## 4. Configure and run

Everything about *how* to simulate (time step, end time, integrator, reference frame,
which network model, what to print) is a `Config` object, the same pydantic model the
GUI renders as its options dialog, and `simulate` takes any of its fields as a
keyword. The fields used in this tutorial: `T_end`, `ts` (output step),
`line_dyn` (section 9), `incl_lim` (limiters), `omega_mode` (`"nom"`, `"coi"`,
`"single"`, `"dist"` reference frame), `int_scheme_sim` (`"idas"`, `"cvodes"`,
`"collocation"`, `"rk"`) with `int_scheme_sim_options` passed straight to CasADi,
`skip_disturance` (spelled like that upstream) for the pure steady state,
`small_signal_analysis`, `parametric` (notebook 02), `quiet`. `run` (from the
setup cell) only pre-fills the workshop settings `WS`.

`simulate` parsed the files, built the DAE symbolically with CasADi, solved the
initial power flow, initialized every device at steady state, and integrated
through the load step with an implicit DAE solver (IDAS). The returned object
holds the symbolic model *and* the trajectories. (`power_flow_table(dae)` gives
the operating point it started from as a DataFrame, per bus or per branch.)

## 5. The one-line diagram

`plot_system` draws the parsed system: a synchronous machine is a circle with `∼`, a converter a split
square (DC on one side, AC on the other), a load an arrow, an SVC a diamond, and
every device is labeled with its model type and rating. The same call colors the
buses by any signal at a chosen time (`color_by="bus*:v", at=1.02`), writes the
branch flows on the lines (`annotate_branches=True`), and takes your own
coordinates (`pos=`).

## 6. Every quantity has an address

Signals are addressed as `owner:quantity`. Owners are the devices (by the `idx`
you gave them), the buses (`bus1`, `bus2`, ...) and the branches (`line1-2`, ...).
Quantities are that owner's states plus derived ones: `f` in Hz, `P` in MW and
`Q` in MVAr for devices, `v` / `theta` / `P` / `Q` per bus, `i` / `P` / `Q` per
branch. `signals(dae)` lists them:

Every plotting and export function takes the same selector: one name
(`"SG1:omega"`), a bare quantity applied to every owner that has it (`"f"`), a
glob (`"bus*:v"`), or a list of these. Signals sharing a unit share a panel, so
a mixed selection gives one panel per unit. (`get(dae, ...)` returns the same
selection as arrays, and `plot_states(dae, "GFMI2")` is the small-multiples view
of every state of one device, useful when debugging a model.)

Reading the three panels:

* both units slow down after the step; the converter reacts faster (no inertia, a
  first-order power filter and a droop) and picks up most of the extra load at first,
  the machine's turbine follows through its governor,
* the frequency settles below 50 Hz: droop control shares the load but does not
  restore frequency (that is secondary control),
* the short spike in the voltages right at 1 s is the electromagnetic transient of the
  network, which is only there because the lines are modeled dynamically.

## 7. Metrics and export

`metrics` gives the excursion summary of any set of signals (default: every
frequency): the value before the event, the worst excursion and when it happened,
the final value, and the largest rate of change.

The trajectories leave as a DataFrame (`to_dataframe`) or a CSV (`to_csv`) with `t`
plus one column per signal; `every=n` keeps every n-th sample. For a paper figure,
export the columns the figure needs and plot them with your own tools.

## 8. Change the disturbance

`sim_dist.txt` accepts six event types plus a setpoint step:

* `LOAD`: step a load by `p_delta` / `q_delta` MW/MVAr,
* `FAULT_BUS` / `CLEAR_FAULT_BUS`: a three-phase fault (admittance `y`) at a bus,
* `FAULT_LINE` / `CLEAR_FAULT_LINE`: the same on a line, quasi-static network only
  (with `line_dyn=True` the simulator skips them with a warning; use `FAULT_BUS`),
* `OPEN_LINE`: open a branch permanently,
* `SETPOINT`: step any device setpoint (`device`, `param`, `value`) at a given time.

`set_disturbances` overwrites the file in your local copy, and `read_events` reads
it back. Below, a three-phase short circuit at the load bus (fault admittance
20 p.u.) is applied at 1.0 s and cleared after 100 ms, and `compare` puts the
fault run and the load step on the same panels. The fault run has the deeper
frequency nadir, and unlike the load step it returns to 50 Hz once the fault is
cleared, because no load changed.

In [ ]:
FAULT = 'Disturbance, time = 1.0, type = "FAULT_BUS",       bus = "2", y = 20'
CLEAR = 'Disturbance, time = 1.1, type = "CLEAR_FAULT_BUS", bus = "2"'

In [ ]:
# Put the load step back for the rest of the notebook.
set_disturbances(root, SYS, ['Disturbance, time = 1.0, type = "LOAD", bus = "2", p_delta = 10, q_delta = 0'])

## 9. Hybrid in one flag: quasi-static vs dynamic network

With `line_dyn=True` (what we used so far) every line carries its own current states
and the bus voltages are differential: the electromagnetic transients of the network
are kept. With `line_dyn=False` the network is the usual algebraic admittance matrix of
phasor (RMS) simulation. Same devices, same files, one flag. The time axis is
zoomed onto the switching instant, where the two models differ.

The electromechanical response is the same; the difference is the fast network
transient in the first tens of milliseconds. Use the dynamic network when converter
controls interact with the network at those time scales (inner current loops, PLLs,
LCL filters), the quasi-static one when you want speed on large systems.

hermess printed a warning at that run: the converter's LCL filter kept its fast
current and voltage states while the network they interact with became algebraic,
which is inconsistent (the filter now reacts to a network that responds
instantly). It does not affect the slow comparison above, but a consistent
quasi-static setup swaps the filter strategy too, one word in the file, as
notebook 02 does for the control strategies. `set_param` edits one device row in
the local copy; all three runs agree electromechanically.

In [ ]:
set_param(root, SYS, "GFMI2", filter='"LCL_static"')
dae_rms_qs = run(SYS, system_root=root, T_end=5.0, line_dyn=False)
set_param(root, SYS, "GFMI2", filter='"LCL"')       # back to the shipped filter
print("GFMI2 states with LCL_static:", len(get_device(dae_rms_qs, "GFMI2").states),
      "vs dynamic LCL:", len(get_device(dae_rms, "GFMI2").states))

axs = compare({"EMT": dae, "RMS, dynamic LCL": dae_rms, "RMS, LCL_static": dae_rms_qs},
        "GFMI2:f", ncols=1, figsize=(7, 3.4));

axs[0,0].set_xlim(0.98, 1.02)
axs[0,0].set_ylim(49.995,50.001)

One requirement to know before you switch your own system to dynamic lines: the
line charging acts as the bus capacitance, so every bus needs a connected branch
with `b > 0`. The shipped systems satisfy this; if yours does not, setup stops
with the names of the offending buses and the remedy, instead of failing inside
the integrator.

## 10. Small-signal analysis in one call

Because the DAE is symbolic, the Jacobian at the operating point is exact and needs
no extra modeling.
`modal_table` runs the eigenvalue analysis on the run you already did and lists the
modes, least damped first; `n`, `min_freq` / `max_freq` and `oscillatory_only`
select what you see.

What the table says (the state names read `device@bus:state`):

* the slow part has one electromechanical mode, machine speed and angle swinging against
  the converter angle, at about 0.4 Hz with a damping ratio near 0.5, and an excitation
  mode (field flux and `Efd`) close to it,
* the converter's own angle and power-measurement modes are almost critically damped,
* the least damped modes overall (drop `max_freq`) are at kHz: network and LCL-filter
  resonances that exist only because the lines and the filter are modeled dynamically.

`participation_table` names the states behind one mode, here the least damped slow
one (the excitation mode), and `state_matrix(dae)` returns the reduced state
matrix itself when you need it.

`plot_modes` is the s-plane view (imaginary axis in Hz, dotted rays at the reference
damping); `fmax` and `xlim` zoom onto the slow modes and `annotate=True` writes the
mode ids from the table. Called twice on one axes with `label` and `color`, it
overlays two designs or two network models.

## 11. Turn a knob: droop gain and inertia

`set_param` edits one device row in the local copy (numbers stay numbers, strings
keep their quotes, so `set_param(root, SYS, "GFMI2", angle='"VSM"')` would swap a
control strategy). The slider reruns the simulation (well under a second) and
redraws the frequency response next to the slow eigenvalues. A larger droop gain
`Kp` means more frequency deviation per MW, and a smaller inertia `H` a deeper,
faster first swing.

In [ ]:
from ipywidgets import interact, FloatSlider

def explore(Kp=0.01, H=6.5):
    set_param(root, SYS, "GFMI2", Kp=Kp)
    set_param(root, SYS, "SG1", H=H)
    d = run(SYS, system_root=root, T_end=5.0)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    plot_frequency(d, ax=axs[0]); axs[0].set_ylim(49.85, 50.02)
    plot_modes(d, ax=axs[1], fmax=5, xlim=(-8, 0.5))
    fig.tight_layout(); plt.show()

interact(explore,
         Kp=FloatSlider(value=0.01, min=0.005, max=0.08, step=0.005, continuous_update=False, readout_format=".3f"),
         H=FloatSlider(value=6.5, min=2.0, max=12.0, step=0.5, continuous_update=False));

In [ ]:
# Restore the shipped values so later cells (and notebook 02) start from the same point.
set_param(root, SYS, "GFMI2", Kp=0.01)
set_param(root, SYS, "SG1", H=6.5)

## 12. Beyond the notebook: the command line, and results that travel

The same engine is available from the shell. `hermess list` names the systems,
`hermess run` simulates one, and `--set KEY=VALUE` reaches any `Config` field
(plus `--t-end`, `--ts`, `--small-signal`, `--system-root`, `--no-plot`).
Useful for batch jobs and for checking an install.

In [ ]:
!hermess run 3bus_loadstep --t-end 2 --ts 0.01 --no-plot --set line_dyn=False

And `extract_results` gives the plain-data container the GUI itself receives,
detached from the symbolic model: trajectories (states and, per device, the
algebraic and derived signals the GUI marks "(algebraic)"), power flow tables,
small-signal results, the resolved config, the version and the date. Plain numpy
and pandas, picklable, so it can be saved, shared or archived next to the paper.

In [ ]:
import pathlib, pickle

res = hermess.extract_results(dae, dae.cfg)
pathlib.Path("run_3bus.pkl").write_bytes(pickle.dumps(res))
res2 = pickle.loads(pathlib.Path("run_3bus.pkl").read_bytes())

print(f"system {res2.system!r}, hermess {res2.hermess_version}, created {res2.created}")
print(f"{len(res2.t)} samples, {len(res2.devices)} device units, "
      f"small-signal: {res2.small_signal is not None}")
print("provenance, from the resolved config:",
      {k: res2.config[k] for k in ("T_end", "ts", "line_dyn", "int_scheme_sim")})

## Recap

* A system is two text files; a run is a `Config` and one call. The GUI is the same
  engine behind buttons (its builder writes those files), so everything it showed
  is one call here.
* Disturbances, the network model, and the reference frame are configuration, not code.
* Every quantity has an address, so plotting and exporting are one line each.
* The symbolic DAE gives the small-signal analysis of the same model without extra modeling.
* Results leave as plain data: CSV for figures, a pickled container for archives,
  the CLI for batch work.
* Everything used here ships with the package (`hermess.analysis`); the only
  notebook-side pieces are the ETH style and the `WS` defaults in the setup cell.

Notebook 02 uses the same system for what the symbolic model adds:
writing a new converter control strategy in a few lines, computing exact
sensitivities, and tuning a controller by gradient descent through the simulator.

---
## Appendix: the API on one page

Every function below is `hermess.analysis` and documents its full signature in
its docstring: type `plot?` in a cell, press Shift+Tab inside the brackets, or
call `help(plot)`. The docs site (maitrayadesai.github.io/hermess, page "Analysis
and plotting") carries the same material next to the public API page and the
per-model reference with equations and symbol tables. Three places to look
things up:

* what can go inside `simulate(...)` (and therefore `run(...)`): the `Config`
  fields, listed by `list(hermess.config.Config.model_fields)`; the GUI options
  dialog documents each one with a tooltip, and the CLI validates
  `--set KEY=VALUE` against the same list,
* what a device row in `sim_param.txt` accepts: the model's docstring and its
  docs page (generated from it), or the GUI builder's parameter form,
* what a strategy must implement: the abstract class, e.g.
  `help(hermess.devices.inverter_angle.AngleSource)` (notebook 02 walks it).

| Function | What it does | Section |
|---|---|---|
| `simulate(system, **config)` | run a system; any `Config` field as a keyword, `quiet=`, `progress_callback=`, `init_callback=` | 2, 4 |
| `WS` + `run(system, **overrides)` | this notebook's three-line shim over `simulate` | 0, 4 |
| `list_systems(root=None)`, `hermess.SYSTEMS_DIR`, `show_system(root, name)`, `copy_system(name)` | the shipped systems, where they live, the two files printed, a local editable copy | 1, 3 |
| `summary(dae)`, `power_flow_table(dae, which)` | run summary, initial power flow per bus or branch | 4 |
| `plot_system(dae, ...)` | one-line diagram; `device_labels`, `color_by` + `at`, `annotate_branches`, `pos`, `colors` | 5 |
| `signals(dae, what, kind)`, `signal_names`, `get`, `get_device` | the address book, the resolved names, the arrays, the device object | 6 |
| `plot(dae, what, ...)` | any selector; `ax`, `figsize`, `title`, `events`, `label_prefix`, style kwargs | 6 |
| `plot_frequency` / `plot_voltages` / `plot_active_power`, `plot_states` | the named views; small multiples of a device's states | 6, 11 |
| `metrics(dae, what, settle_from)` | excursion summary | 7 |
| `to_dataframe` / `to_csv(dae, path, what, every, float_format)` | trajectories as a table or a file | 7 |
| `set_disturbances(root, name, rows)`, `read_events` | rewrite and read `sim_dist.txt` | 8 |
| `compare(runs, what, ncols, figsize, colors)` | several runs on the same panels | 8, 9 |
| `set_param(root, name, idx, **values)` | edit one device row; quoted strings swap strategies | 9, 11 |
| `modal_table(dae, n, min_freq, max_freq, oscillatory_only)` | the modes, least damped first | 10 |
| `participation_table(dae, mode, top)`, `small_signal(dae, report=True)`, `state_matrix(dae)` | one mode's states, the printed report, the matrix | 10 |
| `plot_modes(dae, fmax, xlim, annotate, label, color, damping_ref)` | the s-plane, overlays via `ax` | 10, 11 |
| `hermess` CLI, `hermess.extract_results(dae, cfg)` | `list`, `run`, `--set`; the picklable results container | 12 |